In [71]:
import os
import logging
from datetime import datetime

import pandas as pd

In [83]:
# Run params
TS      = datetime.now().strftime("%Y%m%d_%H%M%S")
CLF     = "RF" # name of classification model
SDG     = "none" # name of used generation model 
RUN     = "test" # status of run test or final run 
DATASET = "yeast" # which dataset was used

In [84]:
# set up logging
logger = logging.getLogger(__name__)
for handler in logger.handlers[:]:
    logger.removeHandler(handler)
formatter = logging.Formatter("%(levelname)s: %(message)s")
logger.setLevel(logging.INFO)

# log to stdout
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(formatter)
logger.addHandler(console_handler)

# log to file
log_file = f"../logs/{TS}_{RUN}_{SDG}_{CLF}_{DATASET}"
file_handler = logging.FileHandler(log_file)
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

In [85]:
logger.info("-"*50)
logger.info(f"Timestamp: {TS}")
logger.info(f"Dataset: {DATASET}")
logger.info(f"Synthetic Data Generation Algo: {SDG}")
logger.info(f"Classification Model: {CLF}")
logger.info(f"Type of Run: {RUN}")

INFO: --------------------------------------------------
INFO: Timestamp: 20241118_110843
INFO: Dataset: yeast
INFO: Synthetic Data Generation Algo: none
INFO: Classification Model: RF
INFO: Type of Run: test


In [140]:
df = pd.read_csv("../data/raw/yeast.csv")

In [141]:
data = df[(df["localization_site"] == "CYT") | (df["localization_site"] == "ME3")]

In [147]:
class_maj_grp = data["localization_site"].value_counts().values[0]
class_min_grp = data["localization_site"].value_counts().values[1]

In [153]:
class_min_grp * 100 / (class_maj_grp + class_min_grp)

np.float64(26.038338658146966)

In [151]:
data["localization_site"].value_counts(normalize=True)

localization_site
CYT    0.739617
ME3    0.260383
Name: proportion, dtype: float64

In [156]:
logger.info("-"*50)
logger.info("Dataset infos")
logger.info(f"Class imbalance ratio: {class_min_grp * 100 / (class_maj_grp + class_min_grp)}")
logger.info(f"# Majority class: {class_maj_grp}")
logger.info(f"# Minority class: {class_min_grp}")

INFO: --------------------------------------------------
INFO: Dataset infos
INFO: Class imbalance ratio: 26.038338658146966
INFO: # Majority class: 463
INFO: # Minority class: 163


In [28]:
data

,mcg,gvh,alm,mit,erl,pox,vac,nuc,localization_site
3,0.58,0.44,0.57,0.13,0.5,0.0,0.54,0.22,NUC
5,0.51,0.40,0.56,0.17,0.5,0.5,0.49,0.22,CYT
7,0.48,0.45,0.59,0.20,0.5,0.0,0.58,0.34,NUC
9,0.40,0.39,0.60,0.15,0.5,0.0,0.58,0.30,CYT
10,0.43,0.39,0.54,0.21,0.5,0.0,0.53,0.27,NUC
...,...,...,...,...,...,...,...,...,...
1477,0.38,0.32,0.64,0.41,0.5,0.0,0.44,0.11,CYT
1478,0.38,0.40,0.66,0.35,0.5,0.0,0.43,0.11,CYT
1480,0.47,0.43,0.61,0.40,0.5,0.0,0.48,0.47,NUC
1482,0.43,0.40,0.60,0.16,0.5,0.0,0.53,0.39,NUC


In [64]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import numpy as np

In [41]:
X = data.iloc[:, :-1]
y = data["localization_site"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split()

In [88]:
logger.info("-"*50)
logger.info("Shape of train and test data")
logger.info(f"Timestamp: {datetime.now().strftime('%Y%m%d_%H%M%S')}")
logger.info(f"X_train: {X_train.shape}")
logger.info(f"y_train: {y_train.shape}")
logger.info(f"X_test: {X_test.shape}")
logger.info(f"y_test: {y_test.shape}")

INFO: --------------------------------------------------
INFO: Shape of train and test data
INFO: Timestamp: 20241118_111219
INFO: X_train: (713, 8)
INFO: y_train: (713,)
INFO: X_test: (179, 8)
INFO: y_test: (179,)


In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(713, 8)
(713,)
(179, 8)
(179,)


In [89]:
logger.info("-"*50)
logger.info(f"Stating training of classifier: {datetime.now().strftime('%Y%m%d_%H%M%S')}")

INFO: --------------------------------------------------
INFO: Stating training of classifier: 20241118_111334


In [49]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)

In [ ]:
logger.info("Hyperparameter of Classification Model:")
logger.info(f"{rf_clf.get_params()}")

INFO: Hyperparameter of Classification Model:
INFO: {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


In [50]:
rf_clf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [95]:
logger.info(f"Finished training of classifier: {datetime.now().strftime('%Y%m%d_%H%M%S')}")

INFO: Finished training of classifier: 20241118_111823


In [51]:
y_pred = rf_clf.predict(X_test)

In [ ]:
logger.info("-"*50)
logger.info("Evaluations Metrics")
logger.info(f"Accuracy Score: {accuracy_score(y_test, y_pred)}")
logger.info("Classification Report:")
logger.info(f"{classification_report(y_test, y_pred)}")

INFO: --------------------------------------------------
INFO: Evaluations Metrics
INFO: Accuracy Score: 0.7262569832402235
INFO: Classification Report:
INFO:               precision    recall  f1-score   support

         CYT       0.73      0.74      0.74        93
         NUC       0.72      0.71      0.71        86

    accuracy                           0.73       179
   macro avg       0.73      0.73      0.73       179
weighted avg       0.73      0.73      0.73       179



In [68]:
f1_score(y_test, y_pred, pos_label='CYT')

np.float64(0.7379679144385026)

In [69]:
f1_score(y_test, y_pred, pos_label='NUC')

np.float64(0.7134502923976608)

In [98]:
logger.info("-"*50)
logger.info("Saving results...")

INFO: --------------------------------------------------
INFO: Saving results...


In [99]:
import pickle

In [ ]:
# read pkl file
df = pd.read_pickle("../results/result_df_log.pkl")

In [131]:
df

,ID,Timestamp,RUN,Dataset,SDG,CLF,Accuracy,F1-Score
0,1,20241118_110843,test,yeast,none,RF,0.726257,0.71345


In [123]:
# get id
ID_new = df["ID"].values[0] + 1

In [124]:
ID_new

np.int64(2)

In [133]:
# create dataframe with results
# init
df_new_run = pd.DataFrame(
    [{"ID":ID_new, 
     "Timestamp": TS, 
     "RUN": RUN,
     "Dataset": DATASET,
     "SDG": SDG, 
     "CLF": CLF, 
     "Accuracy": accuracy_score(y_test, y_pred),
     "F1-Score": f1_score(y_test, y_pred, pos_label="NUC")}]
    )

In [134]:
df_new_run

,ID,Timestamp,RUN,Dataset,SDG,CLF,Accuracy,F1-Score
0,2,20241118_110843,test,yeast,none,RF,0.726257,0.71345


In [135]:
df_save = pd.concat([df, df_new_run])

In [136]:
df_save

,ID,Timestamp,RUN,Dataset,SDG,CLF,Accuracy,F1-Score
0,1,20241118_110843,test,yeast,none,RF,0.726257,0.71345
0,2,20241118_110843,test,yeast,none,RF,0.726257,0.71345


In [ ]:
# save to pickle file 
df_save.to_pickle('../results/result_df_log.pkl')

In [ ]:
df_save.to_csv("../results/result_df_log.csv", index=False)

In [106]:
logger.info("Results saved")

INFO: Results saved
